# Breast Cancer Detection Using Deep Learning (CBIS-DDSM)

This notebook trains and compares multiple CNN architectures (transfer learning) for **benign vs malignant** classification on mammography images.

**What this notebook is designed for**
- Clean, reproducible training loop
- Fair comparison across model backbones
- Metrics that make sense for medical classification (not just accuracy)

> Note: The dataset is **not included** in this repository due to licensing restrictions. You must download CBIS-DDSM from the official source and point the notebook to your local dataset path.


## 1) Setup

If you’re running this in Colab, you may want to switch runtime to GPU:
`Runtime → Change runtime type → GPU`.


In [ ]:
import os
import json
import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report, roc_curve
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print('TensorFlow:', tf.__version__)
print('GPUs:', tf.config.list_physical_devices('GPU'))


## 2) Data configuration

This notebook expects your data to be arranged in a simple folder format:

```
DATA_ROOT/
  train/
    benign/
    malignant/
  val/
    benign/
    malignant/
  test/
    benign/
    malignant/
```

If your CBIS-DDSM download is not in this format, you can still use it—just convert it once (copy files into folders) or write a small mapping script.

**Tip:** Keep splits *patient-level* if you have patient IDs available to avoid leakage.


In [ ]:
# TODO: Change this to your local path
DATA_ROOT = r"/path/to/your/cbis-ddsm-splits"

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_dir = os.path.join(DATA_ROOT, "train")
val_dir   = os.path.join(DATA_ROOT, "val")
test_dir  = os.path.join(DATA_ROOT, "test")

for d in [train_dir, val_dir, test_dir]:
    print(d, 'exists:', os.path.isdir(d))


## 3) Build tf.data pipelines

We use `image_dataset_from_directory` because it’s simple and reproducible. The data is cached/prefetched for better performance.


In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    labels='inferred',
    label_mode='binary',
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=SEED,
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    val_dir,
    labels='inferred',
    label_mode='binary',
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    labels='inferred',
    label_mode='binary',
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

class_names = train_ds.class_names
print('Classes:', class_names)

def configure(ds):
    return ds.cache().prefetch(buffer_size=AUTOTUNE)

train_ds = configure(train_ds)
val_ds   = configure(val_ds)
test_ds  = configure(test_ds)


## 4) Quick sanity-check: visualize samples


In [ ]:
plt.figure(figsize=(10, 8))
for images, labels in train_ds.take(1):
    for i in range(9):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        lbl = int(labels[i].numpy()[0])
        plt.title(class_names[lbl])
        plt.axis("off")
plt.tight_layout()
plt.show()


## 5) Model factory (transfer learning)

We’ll standardize:
- Input size
- Optimizer
- Loss
- Training schedule

So differences you see in results are more likely due to the backbone choice.


In [ ]:
DATA_AUG = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.03),
    layers.RandomZoom(0.05),
], name='augmentation')

BACKBONES = {
    'VGG16': keras.applications.VGG16,
    'VGG19': keras.applications.VGG19,
    'ResNet50': keras.applications.ResNet50,
    'ResNet101': keras.applications.ResNet101,
    'InceptionV3': keras.applications.InceptionV3,
    'InceptionResNetV2': keras.applications.InceptionResNetV2,
    'MobileNet': keras.applications.MobileNet,
    'DenseNet121': keras.applications.DenseNet121,
}

def build_model(backbone_name: str, lr: float = 1e-4, dropout: float = 0.3):
    Backbone = BACKBONES[backbone_name]
    base = Backbone(
        include_top=False,
        weights='imagenet',
        input_shape=IMG_SIZE + (3,),
    )
    base.trainable = False  # freeze for initial training

    inputs = keras.Input(shape=IMG_SIZE + (3,))
    x = DATA_AUG(inputs)
    # Preprocessing for each backbone
    preprocess = getattr(keras.applications, backbone_name).preprocess_input
    x = preprocess(x)
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(dropout)(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)
    model = keras.Model(inputs, outputs, name=f'{backbone_name}_binary')

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss='binary_crossentropy',
        metrics=[
            keras.metrics.BinaryAccuracy(name='accuracy'),
            keras.metrics.AUC(name='auc')
        ]
    )
    return model


## 6) Training utilities

We’ll use early stopping and save the best weights.


In [ ]:
from pathlib import Path

OUT_DIR = Path('results')
OUT_DIR.mkdir(exist_ok=True)

def callbacks_for(name: str):
    ckpt_path = OUT_DIR / f'{name}.keras'
    return [
        keras.callbacks.ModelCheckpoint(
            filepath=str(ckpt_path),
            monitor='val_auc',
            mode='max',
            save_best_only=True,
            verbose=1,
        ),
        keras.callbacks.EarlyStopping(
            monitor='val_auc',
            mode='max',
            patience=4,
            restore_best_weights=True,
            verbose=1,
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor='val_auc',
            mode='max',
            factor=0.5,
            patience=2,
            min_lr=1e-6,
            verbose=1,
        )
    ]

def plot_history(history, title='Training curves'):
    hist = pd.DataFrame(history.history)
    plt.figure(figsize=(8, 4))
    if 'accuracy' in hist and 'val_accuracy' in hist:
        plt.plot(hist['accuracy'], label='train_acc')
        plt.plot(hist['val_accuracy'], label='val_acc')
    plt.title(title)
    plt.xlabel('Epoch')
    plt.legend()
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(8, 4))
    if 'auc' in hist and 'val_auc' in hist:
        plt.plot(hist['auc'], label='train_auc')
        plt.plot(hist['val_auc'], label='val_auc')
    plt.title(title.replace('curves', 'AUC'))
    plt.xlabel('Epoch')
    plt.legend()
    plt.tight_layout()
    plt.show()


## 7) Evaluation helpers

We compute classic classification metrics and plot ROC. We also store everything in a results table so the comparison is easy.


In [ ]:
def get_labels_and_probs(model, ds):
    y_true = []
    y_prob = []
    for x, y in ds:
        p = model.predict(x, verbose=0).reshape(-1)
        y_prob.extend(p.tolist())
        y_true.extend(y.numpy().reshape(-1).tolist())
    y_true = np.array(y_true).astype(int)
    y_prob = np.array(y_prob)
    return y_true, y_prob

def evaluate_binary(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    return {
        'accuracy': float(accuracy_score(y_true, y_pred)),
        'precision': float(precision_score(y_true, y_pred, zero_division=0)),
        'recall': float(recall_score(y_true, y_pred, zero_division=0)),
        'f1': float(f1_score(y_true, y_pred, zero_division=0)),
        'roc_auc': float(roc_auc_score(y_true, y_prob)) if len(np.unique(y_true)) > 1 else float('nan'),
    }

def plot_roc(y_true, y_prob, title='ROC Curve'):
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    auc = roc_auc_score(y_true, y_prob)
    plt.figure(figsize=(6, 5))
    plt.plot(fpr, tpr, label=f'AUC={auc:.3f}')
    plt.plot([0, 1], [0, 1], linestyle='--')
    plt.title(title)
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.legend(loc='lower right')
    plt.tight_layout()
    plt.show()

def print_confusion(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    cm = confusion_matrix(y_true, y_pred)
    print('Confusion Matrix (rows=true, cols=pred):')
    print(cm)
    print('\nClassification Report:')
    print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0))


## 8) Train + compare models

Pick the architectures you want to run. Training all 8 backbones can take time.

**Tip:** Start with 2–3 models to confirm your pipeline is working, then scale up.


In [ ]:
MODELS_TO_RUN = [
    'DenseNet121',
    'MobileNet',
    'ResNet50',
]

EPOCHS = 12
LR = 1e-4

results = []

for name in MODELS_TO_RUN:
    print('\n' + '='*80)
    print('Training:', name)
    print('='*80)
    model = build_model(name, lr=LR)
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS,
        callbacks=callbacks_for(name),
        verbose=1,
    )
    plot_history(history, title=f'{name} training curves')

    y_true, y_prob = get_labels_and_probs(model, test_ds)
    metrics = evaluate_binary(y_true, y_prob, threshold=0.5)
    metrics['model'] = name
    results.append(metrics)

    print('Test metrics:', metrics)
    plot_roc(y_true, y_prob, title=f'{name} ROC')
    print_confusion(y_true, y_prob, threshold=0.5)


## 9) Results table

This table is what you’ll paste into your README. Keeping it generated by code avoids copy errors.


In [ ]:
results_df = pd.DataFrame(results)
results_df = results_df[['model','accuracy','precision','recall','f1','roc_auc']].sort_values('roc_auc', ascending=False)
results_df


In [ ]:
out_csv = OUT_DIR / 'metrics_summary.csv'
results_df.to_csv(out_csv, index=False)
print('Saved:', out_csv)


## 10) Optional: Fine-tuning (unfreeze top layers)

Once you confirm your pipeline, you can improve performance by unfreezing part of the backbone and training with a lower learning rate.
This is optional because it increases training time.


In [ ]:
# Example fine-tuning snippet (run after training one backbone)
#
# model = build_model('DenseNet121', lr=1e-4)
# model.fit(...)
#
# # Unfreeze last N layers of the backbone
# backbone = model.get_layer(index=2)  # depends on model graph; adjust if needed
# backbone.trainable = True
# for layer in backbone.layers[:-30]:
#     layer.trainable = False
#
# model.compile(
#     optimizer=keras.optimizers.Adam(learning_rate=1e-5),
#     loss='binary_crossentropy',
#     metrics=[keras.metrics.BinaryAccuracy(name='accuracy'), keras.metrics.AUC(name='auc')]
# )
#
# model.fit(train_ds, validation_data=val_ds, epochs=8, callbacks=callbacks_for('DenseNet121_finetune'))
print('Fine-tuning cell ready (optional).')


## Notes for a recruiter (what I would mention)
- I compared multiple architectures under a consistent pipeline to avoid cherry-picking results.
- I tracked AUC and F1 because accuracy alone can be misleading in medical classification.
- The notebook saves a CSV summary of metrics so the README table stays consistent.
